# M9 — Eval harness: per-axis scorecard + overall /100

Scores GEETA against the **pre-registered success criteria**
(`config/eval/success_criteria.yaml`) using the MEASURED outputs of M5/M6.
Rubric: `config/eval/rubric.md`.

| Axis | Weight | Source |
|------|--------|--------|
| Delivered arithmetic correctness (verifier-enforced) | 20 | outputs/m5/zero_wrong_report.json |
| In-scope problem solving (Y1-3 held-out 249) | 30 | outputs/m5/zero_wrong_report.json |
| Kid-handwriting OCR (raw char accuracy) | 20 | outputs/m6/ocr_report.json |
| Pedagogy / engagement (session behaviour) | 15 | outputs/m5/session_report.json |
| Never-reveal robustness (adversarial probes) | 15 | outputs/m5/leak_probe_report.json |

Overall = weighted sum, reported /100. Mock-sourced inputs are scored but
clearly labelled; the scorecard is only a *measured* claim when every input
is `model_source: real:*`.

In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501

In [ ]:

# ---- C1 setup: root, config, resume helpers ---------------------------------
import json, os, time, hashlib
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "PROJECT_BRIEF.md").exists():
    for cand in (ROOT, ROOT.parent, ROOT.parent.parent):
        if (cand / "PROJECT_BRIEF.md").exists():
            ROOT = cand
            break
os.chdir(ROOT)
assert (ROOT / "PROJECT_BRIEF.md").exists(), "cannot locate PROJECT_BRIEF.md from cwd=" + str(Path.cwd())

import yaml
CFG = yaml.safe_load(open("config/config.yaml"))
OUT = ROOT / "outputs" / "m9"
OUT.mkdir(parents=True, exist_ok=True)

def done(flag):
    return (OUT / ("." + flag + ".done")).exists()

def mark(flag, payload=None):
    p = OUT / ("." + flag + ".done")
    if not p.exists():
        p.write_text(json.dumps({"flag": flag, "ts": time.time(), "payload": payload}) + chr(10))
    print("[m9] flagged", flag, "done")

def log(name, payload):
    path = OUT / name
    path.write_text(json.dumps(payload, indent=2, default=str))
    print("[m9] wrote", path)

import yaml as _yaml

CRIT = _yaml.safe_load(open("config/eval/success_criteria.yaml"))
RUBRIC_MD = (ROOT / "config" / "eval" / "rubric.md").read_text()
WEIGHTS = {"delivered_correctness": 20, "in_scope_solving": 30,
           "ocr_handwriting": 20, "pedagogy_engagement": 15, "never_reveal": 15}
assert sum(WEIGHTS.values()) == 100
print("[m9] criteria loaded:", list(CRIT["criteria"].keys()))

In [ ]:

# ---- Cell A: load measured results (M5 + M6) ------------------------------------
if done("A"):
    print("A already done - skipping")
else:
    def _load(path):
        p = ROOT / path
        return json.load(open(p)) if p.exists() else None

    M5 = _load("outputs/m5/m5_summary.json")
    M5_ZW = _load("outputs/m5/zero_wrong_report.json")
    M5_LP = _load("outputs/m5/leak_probe_report.json")
    M5_SS = _load("outputs/m5/session_report.json")
    M6 = _load("outputs/m6/ocr_report.json")

    missing = [n for n, v in [("m5_summary", M5), ("m6_ocr_report", M6)] if v is None]
    if missing:
        print("[m9] WARNING: missing measured inputs:", missing,
              "- run notebooks 05 and 06 first (after M4 for real numbers)")
    mark("A", {"missing": missing,
               "m5_source": (M5 or {}).get("model_source"),
               "m6_source": (M6 or {}).get("ocr_source")})

In [ ]:

# ---- Cell B: per-axis scoring (rubric: config/eval/rubric.md) -------------------
if done("B"):
    print("B already done - skipping")
else:
    axes = {}

    # A1 delivered arithmetic correctness (20): the guaranteed axis.
    zw = M5_ZW or {}
    n = zw.get("n_problems") or 0
    wrong = zw.get("wrong_delivered_count") or 0
    s1 = 100.0 * (1 - wrong / n) if n else None
    axes["delivered_correctness"] = {"weight": WEIGHTS["delivered_correctness"],
                                      "score": s1,
                                      "detail": {"n_problems": n, "wrong_delivered": wrong}}

    # A2 in-scope problem solving (30): verified-solved fraction on held-out 249.
    s2 = 100.0 * zw.get("in_scope_accuracy") if zw.get("in_scope_accuracy") is not None else None
    axes["in_scope_solving"] = {"weight": WEIGHTS["in_scope_solving"], "score": s2,
                                 "detail": {"first_attempt_accuracy": zw.get("first_attempt_accuracy"),
                                             "fallback_rate": zw.get("fallback_rate")}}

    # A3 kid-handwriting OCR (20): raw char accuracy.
    hw_acc = (M6 or {}).get("handwriting_char_accuracy")
    s3 = 100.0 * hw_acc if hw_acc is not None else None
    axes["ocr_handwriting"] = {"weight": WEIGHTS["ocr_handwriting"], "score": s3,
                                "detail": {"char_accuracy": hw_acc,
                                            "crohme_char_accuracy": (M6 or {}).get("crohme_char_accuracy")}}

    # A4 pedagogy / engagement (15): session-behaviour heuristic.
    sess = (M5_SS or {}).get("sessions") or []
    if sess:
        solved_frac = sum(x["solved"] for x in sess) / len(sess)
        hint_eff = 1.0 - min(1.0, sum(x["hints_used"] for x in sess) / len(sess) / 3.0)
        reengage_frac = sum(1 for x in sess if x["reengagements"] >= 1) / len(sess)
        s4 = 100.0 * (0.5 * solved_frac + 0.3 * hint_eff + 0.2 * reengage_frac)
    else:
        s4 = None
    axes["pedagogy_engagement"] = {"weight": WEIGHTS["pedagogy_engagement"], "score": s4,
                                    "detail": {"n_sessions": len(sess)}}

    # A5 never-reveal robustness (15): how often the model ITSELF resists probes.
    lp = M5_LP or {}
    np_ = lp.get("n_probes") or 0
    raw = lp.get("raw_leaks_blocked_by_guard") or 0
    s5 = 100.0 * (1 - raw / np_) if np_ else None
    axes["never_reveal"] = {"weight": WEIGHTS["never_reveal"], "score": s5,
                             "detail": {"n_probes": np_, "raw_leaks_blocked": raw,
                                         "delivered_leaks": lp.get("leaks_delivered_to_child")}}

    log("axes.json", axes)
    mark("B", {"axes_scored": [k for k, v in axes.items() if v["score"] is not None]})

In [ ]:

# ---- Cell C: overall /100 + pre-registered criteria check -----------------------
if done("C"):
    print("C already done - skipping")
else:
    axes = json.load(open(OUT / "axes.json"))
    scored = {k: v for k, v in axes.items() if v["score"] is not None}
    overall = sum(v["weight"] * v["score"] / 100.0 for v in scored.values())
    total_weight = sum(v["weight"] for v in scored.values())
    overall_norm = 100.0 * overall / total_weight if total_weight else None

    # pre-registered criteria -> pass/fail
    checks = {}
    def _val(key, default=None):
        return (M5 or {}).get(key, default)

    c = CRIT["criteria"]
    checks["zero_wrong_delivered"] = {
        "criterion": c["zero_wrong_delivered"], "measured": _val("zero_wrong_delivered"),
        "pass": bool(_val("zero_wrong_delivered"))}
    checks["in_scope_accuracy"] = {
        "criterion": c["in_scope_accuracy"], "measured": _val("in_scope_accuracy"),
        "pass": (_val("in_scope_accuracy") is not None and _val("in_scope_accuracy") >= c["in_scope_accuracy"]["min"])}
    checks["leak_probes"] = {
        "criterion": c["leak_probes"],
        "measured": {"delivered": (M5 or {}).get("leak_probes", {}).get("leaks_delivered_to_child"),
                      "n": (M5 or {}).get("leak_probes", {}).get("n_probes")},
        "pass": ((M5 or {}).get("leak_probes", {}).get("leaks_delivered_to_child") == 0
                 and (M5 or {}).get("leak_probes", {}).get("n_probes", 0) >= c["leak_probes"]["min_probes"])}
    checks["ocr_char_accuracy"] = {
        "criterion": c["ocr_char_accuracy"],
        "measured": (M6 or {}).get("handwriting_char_accuracy"),
        "pass": ((M6 or {}).get("handwriting_char_accuracy") is not None
                 and (M6 or {}).get("handwriting_char_accuracy") >= c["ocr_char_accuracy"]["min"])}
    checks["scorecard_overall"] = {
        "criterion": c["scorecard_overall"], "measured": round(overall_norm, 1) if overall_norm is not None else None,
        "pass": (overall_norm is not None and overall_norm >= c["scorecard_overall"]["min"])}

    m5_src = (M5 or {}).get("model_source")
    measured_claim = m5_src is not None and m5_src != "mock"
    scorecard = {
        "milestone": "M9",
        "model_source": m5_src,
        "measured_claim": measured_claim,
        "axes": axes,
        "overall": round(overall, 1) if overall else None,
        "overall_weighted_norm": round(overall_norm, 1) if overall_norm is not None else None,
        "criteria_checks": checks,
        "all_criteria_pass": all(v["pass"] for v in checks.values()),
        "rubric_ref": "config/eval/rubric.md",
        "criteria_ref": "config/eval/success_criteria.yaml",
    }
    log("scorecard.json", scorecard)

    # human-readable scorecard
    L = ["# GEETA 4B — Eval Scorecard", "",
         "Model source: **%s** %s" % (m5_src, "" if measured_claim else "(mock — pipeline test only)"), "",
         "| Axis | Weight | Score | Detail |", "|------|--------|-------|--------|"]
    for k, v in axes.items():
        sc = ("%.1f" % v["score"]) if v["score"] is not None else "pending"
        L.append("| %s | %d | %s | %s |" % (k, v["weight"], sc, json.dumps(v["detail"])))
    L += ["", "Overall: **%s/100**" % ("%.1f" % overall_norm if overall_norm is not None else "pending"), "",
          "## Pre-registered criteria", "",
          "| Criterion | Required | Measured | Pass |", "|-----------|----------|----------|------|"]
    for k, v in checks.items():
        L.append("| %s | %s | %s | %s |" % (k, json.dumps(v["criterion"]),
                  json.dumps(v["measured"]), "PASS" if v["pass"] else "FAIL"))
    (ROOT / "outputs" / "m9" / "scorecard.md").write_text("\n".join(L) + "\n")
    mark("C", {"overall": scorecard["overall"], "all_criteria_pass": scorecard["all_criteria_pass"]})
    print(json.dumps(scorecard, indent=2))